# 03 XAI-Erklaerungen

Ziel: globale und lokale Modell-Erklaerungen fuer ausgewaehlte Modelle erstellen. Startpunkt sind Permutation Importance und SHAP. LIME kann spaeter ergaenzt werden.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

In [ ]:
data_path_candidates = [
    Path("DryBeanDataset/Dry_Bean_Dataset.xlsx"),
    Path("../DryBeanDataset/Dry_Bean_Dataset.xlsx"),
]
data_path = next(path for path in data_path_candidates if path.exists())

df = pd.read_excel(data_path)
X_all = df.drop(columns="Class")
y = df["Class"]

selected_features = [
    "Area",
    "Perimeter",
    "AspectRation",
    "Compactness",
    "roundness",
    "ShapeFactor1",
    "ShapeFactor2",
    "ShapeFactor4",
]
X = X_all[selected_features]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

model = RandomForestClassifier(
    n_estimators=300, random_state=RANDOM_STATE, class_weight="balanced"
)
model.fit(X_train, y_train)
f1_score(y_test, model.predict(X_test), average="macro")

## Permutation Importance

Permutation Importance misst, wie stark die Modellleistung sinkt, wenn ein Feature zufaellig vertauscht wird. Dadurch ist die Methode modellunabhaengig und gut als globale Erklaerung geeignet.

In [ ]:
perm = permutation_importance(
    model, X_test, y_test, scoring="f1_macro", n_repeats=20, random_state=RANDOM_STATE
)

perm_df = (
    pd.DataFrame({
        "feature": selected_features,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
)
perm_df

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=perm_df, x="importance_mean", y="feature")
plt.title("Permutation Importance des Random Forest")
plt.xlabel("Rueckgang Macro-F1 bei Permutation")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## SHAP

SHAP wird im naechsten Schritt fuer globale Feature-Wichtigkeit und lokale Einzelerklaerungen verwendet. Falls die Berechnung zu langsam ist, eine Stichprobe aus `X_test` verwenden.

In [ ]:
# Beispiel fuer den naechsten Arbeitsschritt:
# import shap
# explainer = shap.TreeExplainer(model)
# X_sample = X_test.sample(300, random_state=RANDOM_STATE)
# shap_values = explainer.shap_values(X_sample)
# shap.summary_plot(shap_values, X_sample, feature_names=selected_features)